In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
CREATE CATALOG ${catalog};

In [0]:
%sql
CREATE SCHEMA ${catalog}.bronze;
CREATE SCHEMA ${catalog}.silver;
CREATE SCHEMA ${catalog}.gold

In [0]:
%sql
CREATE VOLUME ${catalog}.bronze.volumne;

In [0]:
from pyspark.sql.functions import input_file_name, current_timestamp
from pyspark.sql.functions import lit


volume_path = f'/Volumes/{catalog}/bronze/volumne/'


files = dbutils.fs.ls(volume_path)
file_paths = [f.path for f in files if f.path.endswith('.csv')]

for file_path in file_paths:
    df = spark.read.option("header", "true").csv(file_path)
    file_name= file_path.split('/')[-1].replace('.csv', '').replace('.', '_')
    df = df.withColumn("ingestion_timestamp", current_timestamp()) \
           .withColumn("source_filename", lit(file_name))
    
    
    table_name = file_path.split('/')[-1].replace('.csv', '').replace('.', '_')
    full_table_name = f"{catalog}.bronze.{table_name}"
    
    
    df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)


In [0]:
%sql
SELECT * FROM ${catalog}.bronze.customers;

In [0]:
%sql
SELECT * FROM ${catalog}.bronze.order_items

In [0]:
%sql
select * from ${catalog}.bronze.orders